# Proyecto 2 — V4 (Mejor modelo)

Iteración sobre V3. Mantiene lo que funcionó y añade las palancas con mayor retorno esperado:

1. **Input enriquecido de V3** (`input_final` con title, year, rating predicho, hints, top-5 TF-IDF, plot recortado, tono).
2. **`max_length=512`** (V3 usaba 384, truncando señal útil de los plots largos).
3. **AsymmetricLoss** para multi-label desbalanceado (mejor que BCE con `class_weight` heurístico en V3).
4. **Split iterativo estratificado** (multi-label aware) en vez de `train_test_split` plano.
5. **Ensemble de 3 seeds** de RoBERTa promediados.
6. **Threshold tuning por clase** para maximizar F1 macro (V3 no lo hacía).
7. **Stacking** con TF-IDF + LogReg balanced (errores ortogonales) buscando el `α` que maximice AUC macro en validación.

Objetivo: superar AUC val 0.9232 de V3 y, sobre todo, mejorar el F1 macro (la métrica que más mueve el resultado práctico).

## 0. Dependencias

Mismo kernel `ml` que V3. Ejecuta la celda de abajo para instalar todo lo necesario desde `requirements.txt` (incluye `transformers`, `torch`, `textblob`, `scikit-multilearn`, etc.).

Si ya tienes el entorno listo puedes saltarla.

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import ast
import gc
import os
import re
import shutil
import warnings
from dataclasses import dataclass

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from textblob import TextBlob
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

try:
    from skmultilearn.model_selection import IterativeStratification
    HAS_ITER_STRAT = True
except ImportError:
    HAS_ITER_STRAT = False
    warnings.warn("scikit-multilearn no instalado; se usará train_test_split como fallback")

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("device:", device)

/Users/usermac/Desktop/Repositorios/ml/caso_comparacion_modelos/ml/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## 1. Configuración central

Todo lo configurable vive aquí para iterar rápido. Si tienes poco cómputo: baja `SEEDS` a `[13]` y/o `MAX_LEN` a 384.

In [3]:
@dataclass
class Cfg:
    model_name: str = "roberta-base"  # alternativa: "microsoft/deberta-v3-base" (requiere sentencepiece)
    max_len: int = 512
    train_bs: int = 16
    eval_bs: int = 32
    epochs: int = 5
    lr: float = 1e-5
    weight_decay: float = 0.05
    warmup_ratio: float = 0.06
    seeds: tuple = (13, 42, 2026)
    output_dir: str = "./v4_runs"
    best_dir: str = "./mejor_roberta_v4"  # placeholder, se sobreescribe por seed

cfg = Cfg()
os.makedirs(cfg.output_dir, exist_ok=True)
print(cfg)

Cfg(model_name='roberta-base', max_len=512, train_bs=16, eval_bs=32, epochs=5, lr=1e-05, weight_decay=0.05, warmup_ratio=0.06, seeds=(13, 42, 2026), output_dir='./v4_runs', best_dir='./mejor_roberta_v4')


## 2. Carga de datos

In [4]:
URL = "https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/"
dataTraining = pd.read_csv(URL + "dataTraining.zip", index_col=0)
dataTesting = pd.read_csv(URL + "dataTesting.zip", index_col=0)

def safe_parse_genres(s):
    try:
        return ast.literal_eval(s)
    except Exception:
        return []

dataTraining["genres"] = dataTraining["genres"].apply(safe_parse_genres)

mlb = MultiLabelBinarizer(classes=[
    "Action", "Adventure", "Animation", "Biography", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "Film-Noir", "History",
    "Horror", "Music", "Musical", "Mystery", "News", "Romance",
    "Sci-Fi", "Short", "Sport", "Thriller", "War", "Western",
])
Y_full = mlb.fit_transform(dataTraining["genres"])
print("train:", dataTraining.shape, "test:", dataTesting.shape, "Y:", Y_full.shape)

train: (7895, 5) test: (3383, 3) Y: (7895, 24)


## 3. Construcción de `input_final` (heredado de V3, calculado tras el split para evitar leakage)

Orden de las piezas (los hints van temprano para que RoBERTa los vea en el primer chunk del tokenizer):

```
Movie: {title} Year: {year}. Rating: {predicted_rating}. Hints: {genre_hints}.
Themes: {auto_kw}. Plot: {plot_smart}. Tone: {sent_desc}
```

In [5]:
def add_sentiment_cols(df):
    pol = df["plot"].astype(str).apply(lambda x: TextBlob(x).sentiment.polarity)
    df = df.copy()
    df["sentiment"] = pol
    df["sent_desc"] = pol.apply(lambda x: "dark tone" if x < -0.1 else ("positive tone" if x > 0.1 else "neutral tone"))
    df["sent_round"] = pol.round(1)
    return df

def smart_plot(plot, head_words=150, tail_words=150, threshold=300):
    words = str(plot).split()
    if len(words) <= threshold:
        return " ".join(words)
    return " ".join(words[:head_words]) + " [ETC] " + " ".join(words[-tail_words:])

def build_input_final(df_train, df_val, df_test):
    """Genera input_final para los 3 splits. TF-IDF auxiliar y maps se ajustan SOLO en train."""
    df_train = add_sentiment_cols(df_train)
    df_val   = add_sentiment_cols(df_val)
    df_test  = add_sentiment_cols(df_test)

    # plot_smart
    for df in (df_train, df_val, df_test):
        df["plot_smart"] = df["plot"].apply(smart_plot)

    # TF-IDF auxiliar SOLO sobre train
    aux_tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
    aux_tfidf.fit(df_train["plot"].astype(str))
    feat_names = aux_tfidf.get_feature_names_out()

    def top5_kw(text):
        v = aux_tfidf.transform([str(text)]).toarray().flatten()
        idx = v.argsort()[-5:][::-1]
        return ", ".join(feat_names[i] for i in idx if v[i] > 0)

    for df in (df_train, df_val, df_test):
        df["auto_kw"] = df["plot"].apply(top5_kw)

    # genre hints: mapa palabra -> géneros más frecuentes (solo train)
    word_to_genre = {}
    classes = mlb.classes_
    # Sacar las top palabras tf-idf por género (vector promedio por clase)
    X_train_tfidf = aux_tfidf.transform(df_train["plot"].astype(str))
    Y_train_local = mlb.transform(df_train["genres"]) if "genres" in df_train.columns else None
    if Y_train_local is not None:
        for ci, cname in enumerate(classes):
            mask = Y_train_local[:, ci] == 1
            if mask.sum() == 0:
                continue
            mean_tfidf = np.asarray(X_train_tfidf[mask].mean(axis=0)).flatten()
            top_words = feat_names[mean_tfidf.argsort()[-20:][::-1]]
            for w in top_words:
                word_to_genre.setdefault(w, []).append(cname)

    def hints_for(kw_str):
        if not isinstance(kw_str, str) or not kw_str:
            return "Unknown"
        words = [w.strip() for w in kw_str.split(",")]
        seen, out = set(), []
        for w in words:
            for g in word_to_genre.get(w, []):
                if g not in seen:
                    seen.add(g)
                    out.append(g)
        return ", ".join(out[:3]) or "Unknown"

    for df in (df_train, df_val, df_test):
        df["genre_hints"] = df["auto_kw"].apply(hints_for)

    # rating predicho desde bucket de sentimiento (solo si la columna existe)
    if "rating" in df_train.columns:
        rating_map = df_train.groupby("sent_round")["rating"].agg(lambda x: x.value_counts().index[0]).to_dict()
        default_rating = df_train["rating"].value_counts().index[0]
        for df in (df_train, df_val, df_test):
            df["predicted_rating"] = df["sent_round"].map(rating_map).fillna(default_rating)
    else:
        for df in (df_train, df_val, df_test):
            df["predicted_rating"] = "unknown"

    def make(df):
        return (
            "Movie: " + df["title"].astype(str)
            + " Year: " + df["year"].astype(str)
            + ". Rating: " + df["predicted_rating"].astype(str)
            + ". Hints: " + df["genre_hints"].astype(str)
            + ". Themes: " + df["auto_kw"].astype(str)
            + ". Plot: " + df["plot_smart"].astype(str)
            + ". Tone: " + df["sent_desc"].astype(str)
        )

    df_train["input_final"] = make(df_train)
    df_val["input_final"]   = make(df_val)
    df_test["input_final"]  = make(df_test)

    return df_train, df_val, df_test

## 4. Split iterativo estratificado (80/20)

Mantiene la distribución de las 24 clases en train/val. Si `scikit-multilearn` no está instalado, cae a `train_test_split` plano.

In [6]:
if HAS_ITER_STRAT:
    splitter = IterativeStratification(n_splits=2, order=1, sample_distribution_per_fold=[0.2, 0.8])
    train_idx, val_idx = next(splitter.split(dataTraining.values, Y_full))
else:
    from sklearn.model_selection import train_test_split
    idx = np.arange(len(dataTraining))
    train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

X_train = dataTraining.iloc[train_idx].reset_index(drop=True)
X_val   = dataTraining.iloc[val_idx].reset_index(drop=True)
y_train = Y_full[train_idx]
y_val   = Y_full[val_idx]

# Test viene sin labels
X_test = dataTesting.reset_index().rename(columns={"index": "orig_id"}).copy()
if "orig_id" not in X_test.columns:
    X_test["orig_id"] = X_test.index

X_train, X_val, X_test = build_input_final(X_train, X_val, X_test)
print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)
print("\nEjemplo input_final:")
print(X_train["input_final"].iloc[0][:500])

train: (6311, 13) val: (1584, 13) test: (3383, 12)

Ejemplo input_final:
Movie: How to Be a Serial Killer Year: 2008. Rating: 6.7. Hints: Unknown. Themes: clerk, teach, video, serial, secrets. Plot: a serial killer decides to teach the secrets of his satisfying career to a video store clerk .. Tone: positive tone


## 5. AsymmetricLoss + custom Trainer

Diseñado para multi-label con desbalance fuerte (Ridnik et al., 2021). Penaliza menos los positivos débiles y más los negativos fáciles.

In [7]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, target):
        x_sigmoid = torch.sigmoid(logits)
        xs_pos = x_sigmoid
        xs_neg = 1 - x_sigmoid
        if self.clip and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)
        los_pos = target * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1 - target) * torch.log(xs_neg.clamp(min=self.eps))
        loss = los_pos + los_neg
        pt0 = xs_pos * target
        pt1 = xs_neg * (1 - target)
        pt = pt0 + pt1
        one_sided_gamma = self.gamma_pos * target + self.gamma_neg * (1 - target)
        one_sided_w = torch.pow(1 - pt, one_sided_gamma)
        loss *= one_sided_w
        return -loss.mean()

asym = AsymmetricLoss()

class MultiLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        loss = asym(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

## 6. Dataset y tokenización

In [8]:
class MovieDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.encodings["input_ids"])
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[i], dtype=torch.float)
        return item

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

def tokenize(texts):
    return tokenizer(list(texts), padding="max_length", truncation=True, max_length=cfg.max_len, return_tensors="pt")

train_enc = tokenize(X_train["input_final"])
val_enc   = tokenize(X_val["input_final"])
test_enc  = tokenize(X_test["input_final"])

train_ds = MovieDataset(train_enc, y_train)
val_ds   = MovieDataset(val_enc, y_val)
test_ds  = MovieDataset(test_enc)
print("tokens OK")

tokens OK


## 7. Entrenamiento de ensemble (3 seeds)

Cada seed produce un modelo independiente; promediamos sus `probs` (sigmoide aplicado a los logits del Trainer).

In [9]:
def train_one_seed(seed):
    set_seed(seed)
    run_dir = os.path.join(cfg.output_dir, f"seed_{seed}")
    os.makedirs(run_dir, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=y_train.shape[1],
        problem_type="multi_label_classification",
    ).to(device)

    args = TrainingArguments(
        output_dir=run_dir,
        num_train_epochs=cfg.epochs,
        per_device_train_batch_size=cfg.train_bs,
        per_device_eval_batch_size=cfg.eval_bs,
        learning_rate=cfg.lr,
        weight_decay=cfg.weight_decay,
        warmup_ratio=cfg.warmup_ratio,
        eval_strategy="epoch", 
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=100,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=seed,
    )

    trainer = MultiLabelTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()

    val_logits  = trainer.predict(val_ds).predictions
    test_logits = trainer.predict(test_ds).predictions

    val_probs  = 1.0 / (1.0 + np.exp(-val_logits))
    test_probs = 1.0 / (1.0 + np.exp(-test_logits))

    auc = roc_auc_score(y_val, val_probs, average="macro")
    print(f"[seed {seed}] AUC val = {auc:.4f}")

    # liberar memoria
    del model, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    shutil.rmtree(run_dir, ignore_errors=True)

    return val_probs, test_probs, auc

val_probs_list, test_probs_list, aucs = [], [], []
for s in cfg.seeds:
    vp, tp, au = train_one_seed(s)
    val_probs_list.append(vp)
    test_probs_list.append(tp)
    aucs.append(au)

val_probs_roberta  = np.mean(val_probs_list, axis=0)
test_probs_roberta = np.mean(test_probs_list, axis=0)

auc_ensemble = roc_auc_score(y_val, val_probs_roberta, average="macro")
print(f"\nAUC val individual: {[round(a,4) for a in aucs]}")
print(f"AUC val ENSEMBLE   : {auc_ensemble:.4f}")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 8. Stacking con TF-IDF + LogReg balanced

El baseline de V1 capta n-gramas raros que el transformer puede no aprovechar. Buscamos el blend `α` que maximiza AUC macro en validación.

In [ ]:
def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = re.sub(r"<[^>]+>", " ", t)
    t = t.lower()
    t = re.sub(r"[^a-z\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

for df in (X_train, X_val, X_test):
    df["plot_clean"] = df["plot"].apply(clean_text)

tfidf_stack = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95,
                              max_features=50000, sublinear_tf=True, stop_words="english")
X_tr_tfidf = tfidf_stack.fit_transform(X_train["plot_clean"])
X_va_tfidf = tfidf_stack.transform(X_val["plot_clean"])
X_te_tfidf = tfidf_stack.transform(X_test["plot_clean"])

lr = OneVsRestClassifier(LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", class_weight="balanced"), n_jobs=1)
lr.fit(X_tr_tfidf, y_train)

val_probs_lr  = np.column_stack([est.predict_proba(X_va_tfidf)[:, 1] for est in lr.estimators_])
test_probs_lr = np.column_stack([est.predict_proba(X_te_tfidf)[:, 1] for est in lr.estimators_])

auc_lr = roc_auc_score(y_val, val_probs_lr, average="macro")
print(f"AUC val LR-TFIDF solo: {auc_lr:.4f}")

best_alpha, best_auc = 1.0, auc_ensemble
for alpha in np.arange(0.0, 1.01, 0.05):
    blend = alpha * val_probs_roberta + (1 - alpha) * val_probs_lr
    a = roc_auc_score(y_val, blend, average="macro")
    if a > best_auc:
        best_auc, best_alpha = a, alpha

print(f"Mejor α (peso RoBERTa) = {best_alpha:.2f} -> AUC val = {best_auc:.4f}")

val_probs_final  = best_alpha * val_probs_roberta  + (1 - best_alpha) * val_probs_lr
test_probs_final = best_alpha * test_probs_roberta + (1 - best_alpha) * test_probs_lr

## 9. Threshold tuning por clase (para F1 macro)

Nota: los **thresholds no afectan el AUC** (métrica continua). Sí afectan F1 macro/micro y exact-match, que son las que más mueven la calidad práctica. Los reportamos pero la submission de Kaggle usa probabilidades continuas, no decisiones binarizadas.

In [ ]:
def tune_thresholds(y_true, y_prob, grid=np.arange(0.05, 0.96, 0.01)):
    thr = np.full(y_true.shape[1], 0.5)
    for c in range(y_true.shape[1]):
        best_th, best_f1 = 0.5, -1.0
        for th in grid:
            yp = (y_prob[:, c] >= th).astype(int)
            f1 = f1_score(y_true[:, c], yp, zero_division=0)
            if f1 > best_f1:
                best_f1, best_th = f1, th
        thr[c] = best_th
    return thr

thr = tune_thresholds(y_val, val_probs_final)

y_val_05  = (val_probs_final >= 0.5).astype(int)
y_val_thr = (val_probs_final >= thr).astype(int)

f1m_05  = f1_score(y_val, y_val_05,  average="macro", zero_division=0)
f1m_thr = f1_score(y_val, y_val_thr, average="macro", zero_division=0)
f1u_05  = f1_score(y_val, y_val_05,  average="micro", zero_division=0)
f1u_thr = f1_score(y_val, y_val_thr, average="micro", zero_division=0)

print(f"F1 macro  th=0.5 : {f1m_05:.4f}  |  th tuneado: {f1m_thr:.4f}")
print(f"F1 micro  th=0.5 : {f1u_05:.4f}  |  th tuneado: {f1u_thr:.4f}")
print("\nThresholds por clase:")
for c, t in zip(mlb.classes_, thr):
    print(f"  {c:12s} -> {t:.2f}")

## 10. Resumen de métricas vs. V3

In [ ]:
resumen = pd.DataFrame([
    {"modelo": "V1 (TF-IDF + LR balanced)",                            "AUC val": 0.8977, "F1 macro": 0.569},
    {"modelo": "V3 (RoBERTa input mejorado + calibrado)",              "AUC val": 0.9232, "F1 macro": np.nan},
    {"modelo": "V4 ensemble RoBERTa",                                  "AUC val": float(auc_ensemble), "F1 macro": np.nan},
    {"modelo": f"V4 stacking (α={best_alpha:.2f})",                    "AUC val": float(best_auc),     "F1 macro": float(f1m_thr)},
])
resumen

## 11. Submission de Kaggle

Se entregan probabilidades continuas en `[0, 1]` (no binarizadas). Formato `p_Action` … `p_Western`, `index_label="ID"`.

In [ ]:
output_cols = [f"p_{c}" for c in mlb.classes_]
sub = pd.DataFrame(test_probs_final, columns=output_cols, index=dataTesting.index)
sub.index.name = "ID"
sub_path = "pred_genres_v4_stacking.csv"
sub.to_csv(sub_path)
print("Guardado:", sub_path, sub.shape)
sub.head()

## 12. Persistencia de artefactos

No guardamos el modelo de transformers (cientos de MB; entrenarlo es lo costoso) sino los artefactos del lado clasico para que `api.py` pueda servir el blend si decidimos migrar. El checkpoint de RoBERTa queda en `./v4_runs/` durante la ejecución y se borra al final por seed para ahorrar disco; si quieres conservarlo, comenta el `shutil.rmtree(...)` en la celda de entrenamiento.

In [ ]:
joblib.dump(mlb,         "mlb_genres_v4.pkl")
joblib.dump(output_cols, "output_columns_v4.pkl")
joblib.dump(tfidf_stack, "tfidf_vectorizer_v4.pkl")
joblib.dump(lr,          "genre_clf_lr_v4.pkl")
joblib.dump({"alpha": float(best_alpha), "thresholds": thr.tolist(),
             "auc_val": float(best_auc), "f1_macro_val": float(f1m_thr),
             "seeds": list(cfg.seeds), "model_name": cfg.model_name,
             "max_len": cfg.max_len},
            "v4_metadata.pkl")
print("Artefactos guardados.")